# Cross-lingual PROBE transfer — the GPU session (LP4FM)

**Runtime → GPU (L4). Run all.** One session, roughly an hour.

This is the only GPU step LP4FM needs. Everything else — the surface
baselines, the renaming conditions, the exporter — is already run and
committed in `results/lp4fm/`.

**What it produces.** The 18 probe cells that currently read `not run`. Until
they exist the paper has a model-free lower bound and no claim about what the
model adds.

**The number to beat is 0.95, not majority chance.** A character n-gram model
with the variable name masked already transfers at up to 0.979 across these
languages. If the probe lands near that, cross-lingual role transfer is
surface regularity; if it clears it, there is something in the residual
stream that the surface does not carry. Either result is the paper.

**Costs, computed rather than guessed.** Extraction caches one forward per
program, so the GPU work is ~8,700 forwards, not one per occurrence. Store
size is what actually binds:

| | uncapped | capped (cell 3) |
|---|---|---|
| Qwen2.5-Coder-1.5B, 3 languages | 3.6 GB | ~1.6 GB |
| StarCoder2-7B, 3 languages | 12.4 GB | ~5.4 GB |

Start with the 1.5B model. If the probe matches the baseline, a second model
will not change that conclusion and the GPU is better spent elsewhere.


In [ ]:
# 1 - setup
import pathlib, os
REPO = "/content/mech-interp"
if not pathlib.Path(REPO).exists():
    !git clone -q https://github.com/nolanlwin/mech-interp.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q -B main origin/main && git pull -q
!git log --oneline -1
!pip install -q transformers==5.8.0 torch numpy scikit-learn matplotlib tree_sitter \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1" \
  "tree-sitter-java>=0.23.5" "tree-sitter-cpp>=0.23.4" \
  "tree-sitter-go>=0.25.0" "tree-sitter-ruby>=0.23.1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/mech-interp/crosslang"
!mkdir -p data/xlcost outputs/role_occ outputs/activations_xlcost outputs/crosslang {DEST}
!cp -rn {DEST}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n  {DEST}/role_occ/* outputs/role_occ/ 2>/dev/null || true
!cp -n  {DEST}/data_xlcost/* data/xlcost/ 2>/dev/null || true
import torch
print(f"setup complete | cuda={torch.cuda.is_available()} "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''}")


In [ ]:
# 2 - CONFIG
MODEL = "Qwen/Qwen2.5-Coder-1.5B"
ROLES = ["accumulator", "iterator", "index_key"]
LANGS = {"Python": "python", "Javascript": "javascript", "PHP": "php"}
SPLIT = "train"
MAX_PER_ROLE = 3000        # see the store-size table above
MODEL_SLUG = MODEL.split("/")[-1].lower().replace(".", "").replace("-", "")
print(f"{MODEL}  slug={MODEL_SLUG}\nroles={ROLES}\nlanguages={list(LANGS)}")


In [ ]:
# 3 - corpora, role occurrences, and the cap. CPU, ~10 min.
#     Restricted to problems shared by at least one language PAIR: unmatched
#     transfer confounds "roles do not transfer" with "different problems",
#     so activations for unshared programs would be extracted and never used.
import json, itertools
for L, slug in LANGS.items():
    if not pathlib.Path(f"data/xlcost/{slug}_{SPLIT}.jsonl.stats.json").exists():
        !python scripts/xlcost_data.py build --language "{L}" --split {SPLIT} --out-dir data/xlcost

ids = {slug: {json.loads(l)["problem_id"]
              for l in open(f"data/xlcost/{slug}_{SPLIT}.jsonl")} for slug in LANGS.values()}
shared = set()
for a, b in itertools.combinations(ids, 2):
    shared |= ids[a] & ids[b]
print(f"problems shared by at least one pair: {len(shared)}")

for L, slug in LANGS.items():
    sub = f"data/xlcost/{slug}_{SPLIT}_shared.jsonl"
    with open(sub, "w") as out:
        for line in open(f"data/xlcost/{slug}_{SPLIT}.jsonl"):
            if json.loads(line)["problem_id"] in shared:
                out.write(line)
    occ = f"outputs/role_occ/all_{slug}_{SPLIT}.jsonl"
    if not pathlib.Path(occ + ".stats.json").exists():
        !python scripts/role_occurrences.py extract --input {sub} --role all --output {occ}
    capped = f"outputs/role_occ/capped_{slug}_{SPLIT}.jsonl"
    # ROLES_ARG is built in Python, not inlined as {" ".join(ROLES)}: IPython
    # evaluates braces inside ! commands, and nested quotes there are fragile.
    ROLES_ARG = " ".join(ROLES)
    if not pathlib.Path(capped + ".stats.json").exists():
        !python scripts/cap_occurrences.py --input {occ} --output {capped} \
          --roles {ROLES_ARG} --max-per-role {MAX_PER_ROLE}


In [ ]:
# 4 - THE GPU STEP: activation stores, one per language. ~30-45 min.
#     --label-field role is NOT optional. The store has one label slot named
#     occurrence_type; role_occurrences writes `role`. Without the flag every
#     stored label is null and probe.py drops every record -- after the GPU
#     time is spent. The choice is stamped into meta.json, and resuming with a
#     different value is refused.
for L, slug in LANGS.items():
    store = f"outputs/activations_xlcost/{slug}_{SPLIT}_{MODEL_SLUG}"
    print(f"\n######## {L} ########")
    !python scripts/extract_activations.py run \
      --canonical data/xlcost/{slug}_{SPLIT}_shared.jsonl \
      --occurrences outputs/role_occ/capped_{slug}_{SPLIT}.jsonl \
      --model-id {MODEL} --label-field role \
      --out-dir {store} --log-every 500
    !mkdir -p {DEST}/stores && cp -r {store} {DEST}/stores/ 2>/dev/null || true


In [ ]:
# 5 - the probe transfer matrix. CPU once the stores exist.
import itertools
for role in ROLES:
    for a, b in itertools.permutations(LANGS.values(), 2):
        sa = f"outputs/activations_xlcost/{a}_{SPLIT}_{MODEL_SLUG}"
        sb = f"outputs/activations_xlcost/{b}_{SPLIT}_{MODEL_SLUG}"
        out = f"outputs/crosslang/probe_{role}_{a}_to_{b}.json"
        if pathlib.Path(out).exists():
            continue
        print(f"\n=== {role}: {a} -> {b}")
        !python scripts/crosslang.py run --train-store {sa} --test-store {sb} \
          --role {role} --output {out}


In [ ]:
# 6 - export: probe beside its baseline, in one table.
#     The baseline cells come from results/lp4fm/, already committed; this
#     fills the probe column that currently reads "not run".
!cp -n /content/mech-interp/outputs/crosslang/out_*.json outputs/crosslang/ 2>/dev/null || true
!python scripts/export_crosslang.py --in outputs/crosslang --out results/lp4fm

from IPython.display import Image, display
import glob
print(open("results/lp4fm/SUMMARY.md").read())
for f in sorted(glob.glob("results/lp4fm/heatmap_*.png")):
    print(f); display(Image(f))


In [ ]:
# 7 - save to Drive, and optionally push results to GitHub.
!mkdir -p {DEST}/stores {DEST}/role_occ {DEST}/data_xlcost {DEST}/crosslang
!cp -r outputs/activations_xlcost/* {DEST}/stores/ 2>/dev/null || true
!cp outputs/role_occ/* {DEST}/role_occ/ 2>/dev/null || true
!cp data/xlcost/*_{SPLIT}*.jsonl* {DEST}/data_xlcost/ 2>/dev/null || true
!cp outputs/crosslang/*.json {DEST}/crosslang/ 2>/dev/null || true
print(f"artifacts saved to {DEST}")

# Optional push. GH_TOKEN = fine-grained PAT, this repo only, Contents R/W.
from google.colab import userdata
try:
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
if not tok:
    print("No GH_TOKEN secret - download results/lp4fm/ and commit locally.")
else:
    os.environ["GH_TOKEN"] = tok
    !git config user.email "naingoolwin.astrio@gmail.com"
    !git config user.name "naingoolwin"
    !git add results/lp4fm
    !git commit -q -m "Cross-lingual probe transfer: {MODEL_SLUG}" || echo "nothing to commit"
    !git push -q https://$GH_TOKEN@github.com/nolanlwin/mech-interp.git HEAD:main && echo "pushed"
